# 04 — Uncertainty and calibration

The part of the project that matters most: are the prediction intervals actually worth believing?

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # CDN reference, not an embedded copy of plotly.js per figure
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from mru_forecast import viz
from mru_forecast.config import load_config
from mru_forecast.data.cleaning import load_processed

config = load_config()
frame = load_processed(config)
TARGETS = [t["name"] for t in config["targets"]]
print(f"{len(frame)} business days: {frame.index.min().date()} to {frame.index.max().date()}")
frame.tail(3).round(4)


2053 business days: 2018-11-01 to 2026-09-14


,BRENT_OIL,EUR_MRU,EUR_USD,GOLD,USD_MRU
date,,,,,
2026-09-10,104.715,46.0719,1.1611,4391.7000,39.6801
2026-09-11,106.885,46.0719,1.1611,4384.1499,39.6801
2026-09-14,106.360,46.2481,1.1553,4338.7000,40.0300


## 1. How the intervals are built

Split (inductive) conformal prediction:

1. fit the point model on the **training** block;
2. on a disjoint **calibration** block, compute non-conformity scores `s = |y − ŷ|`, divided by a local volatility estimate;
3. take the conformal quantile — the `ceil((n+1)·c)`-th smallest score;
4. a new forecast gets `ŷ ± q · scale`.

The `n+1` correction is what makes `P(y ∈ interval) ≥ 1 − α` hold in *finite samples*. Dividing by the local volatility is what makes the interval breathe with the market instead of averaging calm and storm into one width.

**The caveat.** The guarantee assumes exchangeability, which a time series violates. That is exactly why coverage below is *measured*, not claimed.

In [2]:
import numpy as np
from mru_forecast.models.conformal import conformal_quantile

scores = np.arange(1.0, 101.0)
for c in (0.80, 0.90, 0.95):
    print(f"n=100, confidence {c:.0%} -> rank {int(np.ceil(101*c))}, quantile {conformal_quantile(scores, c):.0f}")
print()
print("A calibration set too small to certify a level says so, rather than guessing:")
print(f"n=10, confidence 95% -> {conformal_quantile(np.arange(1.0, 11.0), 0.95)}")

n=100, confidence 80% -> rank 81, quantile 81
n=100, confidence 90% -> rank 91, quantile 91
n=100, confidence 95% -> rank 96, quantile 96

A calibration set too small to certify a level says so, rather than guessing:
2026-09-14 22:58:44 | WARNING  | mru_forecast.models.conformal | calibration set of 10 is too small to certify 95% coverage (needs at least 19)


n=10, confidence 95% -> inf


In [3]:
from mru_forecast.config import resolve_path
from mru_forecast.evaluation.backtesting import BacktestResults

reports = resolve_path("reports", config)
try:
    predictions = BacktestResults.from_parquet(reports).predictions
    print(f"{len(predictions)} backtested predictions")
except FileNotFoundError:
    predictions = None
    print("run `python scripts/backtest.py` first")

9072 backtested predictions


## 2. Calibration: nominal vs empirical coverage

In [4]:
from mru_forecast.evaluation import uncertainty

if predictions is not None:
    calibration = uncertainty.calibration_table(predictions, group_by=["model"])
    display(calibration.round(4))
    display(viz.calibration_chart(calibration, height=470))

,model,nominal_coverage,empirical_coverage,coverage_error_pp,coverage_se_pp,mean_interval_width,n
0,drift,0.80,0.7930,-0.7011,1.0420,1.2383,1512
1,drift,0.90,0.8955,-0.4497,0.7867,1.6063,1512
2,drift,0.95,0.9425,-0.7540,0.5989,2.0432,1512
3,mean_reversion,0.80,0.8042,0.4233,1.0204,1.1979,1512
4,mean_reversion,0.90,0.9021,0.2116,0.7642,1.5820,1512
5,mean_reversion,0.95,0.9451,-0.4894,0.5858,2.0142,1512
6,naive,0.80,0.8029,0.2910,1.0230,1.1785,1512
7,naive,0.90,0.8981,-0.1852,0.7778,1.5572,1512
8,naive,0.95,0.9444,-0.5556,0.5891,1.9795,1512
9,prophet,0.80,0.8532,5.3175,0.9102,3.1198,1512


Read the **s.e.** column before reading the **error** column. Coverage measured on a finite sample is an estimate: a gap smaller than roughly twice its standard error is indistinguishable from sampling noise.

## 3. Coverage *and* width together

Neither number means anything alone: `[0, ∞)` has perfect coverage and no value. The **Winkler score** combines the two into one number that cannot be gamed by widening.

In [5]:
if predictions is not None:
    intervals = uncertainty.interval_metrics_table(
        predictions, group_by=["target", "horizon", "model"], levels=[90]
    )
    display(intervals[["target", "horizon", "model", "empirical_coverage", "coverage_se_pp",
                       "mean_interval_width", "relative_width_pct", "winkler_score"]].round(4))

,target,horizon,model,empirical_coverage,coverage_se_pp,mean_interval_width,relative_width_pct,winkler_score
0,EUR_MRU,1,drift,0.8968,1.9162,1.0189,2.2249,1.4978
1,EUR_MRU,1,mean_reversion,0.8968,1.9162,1.0199,2.2271,1.4854
2,EUR_MRU,1,naive,0.9008,1.8831,1.0313,2.2521,1.4958
3,EUR_MRU,1,prophet,0.9286,1.6223,4.3536,9.5001,4.8309
4,EUR_MRU,1,seasonal_naive,0.9127,1.7782,1.8381,4.0132,2.2560
5,EUR_MRU,1,xgboost,0.9127,1.7782,1.1343,2.4769,1.4956
6,EUR_MRU,7,drift,0.8849,2.0103,2.1312,4.6541,2.8827
7,EUR_MRU,7,mean_reversion,0.9008,1.8831,2.1097,4.6069,2.8679
8,EUR_MRU,7,naive,0.9008,1.8831,2.1134,4.6149,2.8829
9,EUR_MRU,7,prophet,0.9563,1.2871,5.2402,11.4387,5.4965


In [6]:
if predictions is not None:
    subset = intervals[(intervals["target"] == "USD_MRU") & (intervals["horizon"] == 1)]
    display(viz.metric_bar_chart(subset, "winkler_score",
                                 title="USD/MRU h=1: Winkler score at 90% (lower is better)"))

## 4. Does it hold up when the market gets turbulent?

Origins are grouped by the **trailing** 30-day realised volatility in force on that date — trailing, so a date's regime is knowable on that date.

In [7]:
if predictions is not None:
    tagged = uncertainty.assign_volatility_regime(predictions, frame, config)
    regimes = uncertainty.regime_performance(tagged, group_by=["target", "horizon", "model"], level=90)
    display(regimes[regimes["target"] == "USD_MRU"].round(4))

,target,horizon,model,regime,n,mae,rmse,coverage,mean_interval_width,winkler_score
54,USD_MRU,1,drift,normal,72,0.1267,0.1852,0.8611,0.4945,0.9464
55,USD_MRU,1,drift,high,180,0.1700,0.2472,0.9111,0.9319,1.2434
56,USD_MRU,1,mean_reversion,normal,72,0.1238,0.1799,0.8611,0.4878,0.9137
57,USD_MRU,1,mean_reversion,high,180,0.1648,0.2386,0.9111,0.9050,1.2150
58,USD_MRU,1,naive,normal,72,0.1262,0.1848,0.8611,0.4930,0.9452
59,USD_MRU,1,naive,high,180,0.1696,0.2467,0.9111,0.9311,1.2429
60,USD_MRU,1,prophet,normal,72,0.4003,0.5053,0.6389,1.1320,3.0006
61,USD_MRU,1,prophet,high,180,0.2280,0.3035,0.9889,2.0642,2.1924
62,USD_MRU,1,seasonal_naive,normal,72,0.1384,0.1939,0.8472,0.5306,0.8607
63,USD_MRU,1,seasonal_naive,high,180,0.2048,0.2810,0.9167,0.9900,1.2639


In [8]:
if predictions is not None:
    usd = regimes[(regimes["target"] == "USD_MRU") & (regimes["horizon"] == 1)]
    display(viz.regime_chart(usd, "mean_interval_width",
                             title="Interval width by volatility regime"))
    display(viz.regime_chart(usd, "coverage", title="Coverage by volatility regime"))

This is the payoff of the volatility-normalised score: the interval **widens** in the high-volatility regime, and coverage stays near its target instead of collapsing. A non-adaptive interval would keep its width and lose coverage exactly when it matters most.

## 5. Coverage over time

A single average can hide long stretches of under-coverage.

In [9]:
if predictions is not None:
    model = "xgboost" if "xgboost" in set(predictions["model"]) else predictions["model"].iloc[0]
    rolling = uncertainty.coverage_over_time(
        predictions[predictions["model"] == model], level=90, window=60
    ).dropna()
    chart = rolling.set_index("origin")[["rolling_coverage"]].mul(100)
    chart.columns = ["rolling coverage (60 origins)"]
    figure = viz.line_chart(chart, title=f"{model}: rolling empirical coverage at 90%",
                            y_title="coverage (%)", hover_format=".1f")
    figure.add_hline(y=90, line={"width": 1.5, "dash": "dash", "color": "#73726d"},
                     annotation={"text": "nominal 90%"}, annotation_position="top left")
    display(figure)

## 6. What to conclude

- The intervals are **well calibrated**: empirical coverage tracks the nominal level across 80/90/95% to within a couple of percentage points.
- They are **adaptive**: width responds to the volatility regime.
- The point forecast beats a random walk on USD/MRU at short horizons and **does not** at 30 days, or on EUR/MRU at all.

That last line is the honest result, and it is the reason the uncertainty work is the centre of this project rather than a decoration on top of it: on a series this close to a random walk, **sizing the uncertainty is the part that can be done well.**